In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_prod_geo_macro_text = pd.read_csv ('external_data/df_prod_geo_macro_text.csv', low_memory=False)

In [ ]:
df_prod_geo_macro_text.info()

# Подготовка данных для этапа M4 (Structure + Geo + Macro + Text)

In [ ]:
DEDUP_SUBSET = ["text_for_model", "salary_from_log", "region_name"]
n_before = len(df_prod_geo_macro_text)
df_work = df_prod_geo_macro_text.drop_duplicates(
    subset=DEDUP_SUBSET, keep="first"
).reset_index(drop=True)
print("Строк до дедупа:", n_before, "| после:", len(df_work))

In [ ]:

feature_numeric = [
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
    "geo_available",
    "distance_to_reg_center",
    "distance_missing",
    "RK",
    "GRP_K",
    "U_delta",
    "macro_k_u_imputed"
]
feature_categorical = [
    "role_name",
    "schedule_id",
    "employment_id",
    "economic_region",
    "region_name",
    "address.city_new",
    "experience_ord",
    "geohash_4",
    "geohash_5",
    "geohash_6"
]
feature_text = ["text_for_model"]
target = "salary_from_log"
feature_service = ["row_id","salary_from_adj"]
_cols = (
    feature_numeric
    + feature_categorical
    + feature_service
    + feature_text
    + [target]
)
_cols = list(dict.fromkeys(_cols))
df_full = df_work[_cols].copy()
assert df_full["row_id"].is_unique, "row_id не уникален после дедупа"
assert df_full["text_for_model"].astype(str).str.strip().str.len().gt(0).all()

In [ ]:
df_full.to_csv("data_for_models/df_full.csv", index=False)

In [ ]:
# выполним контроль качества витрины df_full
print("Размер выборки и типы данных")
print(df_full.shape)
print(df_full.dtypes)
print("\nПропуски (count / % от строк)")
na_cnt = df_full.isna().sum()
na_pct = (na_cnt / len(df_full) * 100).round(2)
qa_na = pd.DataFrame({"na_count": na_cnt, "na_pct": na_pct})
qa_na = qa_na[qa_na["na_count"] > 0].sort_values("na_count", ascending=False)
display(qa_na if len(qa_na) else "Нет пропусков")
assert df_full[target].notna().all(), "Пропуски в таргете"
assert df_full["region_name"].notna().all(), "Пропуски в region_name (GroupKFold)"
assert (
    df_full["text_for_model"].astype(str).str.strip().str.len() > 0
).all(), "Пустой text_for_model"
assert df_full["row_id"].is_unique, "row_id не уникален"
dup_after = int(df_full.duplicated(subset=DEDUP_SUBSET).sum())
assert dup_after == 0, f"Дубликаты по {DEDUP_SUBSET} в df_full: {dup_after}"
print("\nКардинальность категориальных (для one-hot vs embedding в M4)")
card_rows = []
for col in feature_categorical:
    n_u = df_full[col].nunique(dropna=False)
    card_rows.append({"column": col, "n_unique": int(n_u)})
card_df = pd.DataFrame(card_rows).sort_values("n_unique", ascending=False)
display(card_df)


In [ ]:
#Проверим пропуски
sns.heatmap(df_full.isna().T)
plt.title('Тепловая карта пропусков значений')
plt.show()

In [ ]:
print(df_full.dtypes)

In [ ]:
assert df_full["geo_available"].notna().all()
assert set(df_full["geo_available"].unique()).issubset({0, 1})
df_full["geo_available"] = df_full["geo_available"].astype("float32")

In [ ]:
assert df_full["distance_missing"].notna().all()
assert set(df_full["distance_missing"].unique()).issubset({0, 1})
df_full["distance_missing"] = df_full["distance_missing"].astype("float32")

In [ ]:
assert df_full["macro_k_u_imputed"].notna().all()
assert set(df_full["macro_k_u_imputed"].unique()).issubset({0, 1})
df_full["macro_k_u_imputed"] = df_full["macro_k_u_imputed"].astype("float32")

In [ ]:
df_full.to_csv("data_for_models/df_full.csv", index=False)